<a href="https://colab.research.google.com/github/jianchang512/AIGenerateTool/blob/main/AIGenerateToolColab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Run AIGenerateTool WebUI on Google Colab

This notebook installs [AIGenerateTool](https://github.com/jianchang512/AIGenerateTool) in an isolated Python 3.10 environment and launches its Gradio WebUI. Colab runtimes are temporary, so generated files and configuration are removed when the runtime is reset.

> **Note:** The WebUI only exposes a subset of AIGenerateTool' features (basic video translation, subtitle recognition/translation, TTS). For the full feature set (voice cloning, real-time editing, more API channels), run the desktop client (`sp.py`) locally instead. See [docs/webui.md](docs/webui.md) for details.

## 1. Install AIGenerateTool

The setup is safe to run again: it clones the repository on the first run and updates it on later runs. Dependencies are installed in the project's virtual environment (via `uv`) to avoid conflicts with Colab's preinstalled packages. `AIGenerateTool` requires Python 3.10 specifically.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/vancongquoc121/AIGenerateTool02.git"
REPO_DIR = Path("/content/AIGenerateTool02")

# System packages required by AIGenerateTool (ffmpeg for media processing, libsndfile for audio I/O).
subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg", "libsndfile1-dev"], check=True)

# Update an existing checkout so rerunning this cell does not fail during clone.
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(["python", "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "python", "install", "3.10"], check=True)
# Use the lockfile in an isolated environment without changing Colab's preinstalled packages.
subprocess.run(["uv", "sync", "--frozen", "--extra", "webui", "--python", "3.10"], check=True)
print(f"AIGenerateTool is ready in {REPO_DIR}")

## 2. Launch the WebUI

`webui.py` is built on Gradio, which can create its own public link via `--share`, so no separate tunnel service (e.g. ngrok) is needed. The cell waits for the `*.gradio.live` link to appear in the server log before printing it. If startup fails, it includes the recent server log in the error.

After opening the URL, use the **⚙️ 渠道设置** (Channel Settings) tab to configure the required model and media API keys.

In [ ]:
import subprocess
from pathlib import Path

def main():
    # Đường dẫn thư mục cần chạy
    work_dir = Path("/content/AIGenerateTool02")

    # Lệnh cần thực thi
    command = ["uv", "run", "webui.py", "--share", "--port", "7860"]

    # Chạy lệnh trong thư mục chỉ định
    process = subprocess.Popen(
        command,
        cwd=work_dir,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    # In log ra terminal theo thời gian thực
    for line in iter(process.stdout.readline, b''):
        print(line.decode().strip())

    process.stdout.close()
    process.wait()

if __name__ == "__main__":
    main()

